<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/05-Improve_Prompts_+_Add_Source.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improving RAG: Prompts, Sources, and Response Modes

This notebook builds on the basic RAG pipeline and introduces techniques to
improve response quality: better prompt engineering, source attribution, and
LlamaIndex's built-in response synthesis modes.

**What you'll learn:**
- How to use LlamaIndex's `IngestionPipeline` to chunk and embed documents
- How to store and reload a vector index from ChromaDB
- How to retrieve source nodes with similarity scores
- How LlamaIndex response modes (compact, refine, no_text) affect output
- (Optional) How to write a custom prompt template
- (Optional) How to compare all response modes side-by-side

**Prerequisites:** Familiarity with Python and basic RAG concepts (Notebooks 02–04).

## Setup: Install Packages and Configure API Keys

In [ ]:
!pip install -q llama-index==0.14.19 openai==2.8.1 google-genai==1.70.0 llama-index-embeddings-openai==0.6.0 llama-index-llms-google-genai==0.9.0 \
                opentelemetry-api==1.38.0 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 227.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 47.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 85.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [ ]:
import os

try:
    # In Google Colab: add your keys via the Secrets panel (key icon in the sidebar).
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("API keys loaded from Colab Secrets.")
except ImportError:
    # Running locally — keys are read from environment variables.
    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError("OPENAI_API_KEY is not set.")
    if not os.environ.get("GOOGLE_API_KEY"):
        raise EnvironmentError("GOOGLE_API_KEY is not set.")
    print("API keys loaded from environment variables.")

API keys loaded from Colab Secrets.


In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

# Create a Vector Store


In [ ]:
import chromadb

# PersistentClient saves the vector store to disk at the given path.
# The collection is created here; it will be reloaded in the "Load Indexes" section.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.get_or_create_collection("mini-llama-articles")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  169k  100  169k    0     0   387k      0 --:--:-- --:--:-- --:--:--  387k


## Load the Articles


In [ ]:
import csv

rows = []

# Load the CSV file containing article titles, text, URLs, and source names
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        rows.append(row)

print(f"Loaded {len(rows)} articles from the dataset.")

Loaded 14 articles from the dataset.


# Convert to Document obj


In [ ]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1], metadata={"title": row[0], "url": row[2], "source_name": row[3]}
    )
    for row in rows
]

In [ ]:
print(f"Created {len(documents)} Document objects.")

Created 14 Document objects.


In [ ]:
print(documents[0].text[:500])
print("\nMetadata:", documents[0].metadata)

LLM Variants and Meta's Open Source Before shedding light on four major trends, I'd share the latest Meta's Llama 2 and Code Llama. Meta's Llama 2 represents a sophisticated evolution in LLMs. This suite spans models pretrained and fine-tuned across a parameter spectrum of 7 billion to 70 billion. A specialized derivative, Llama 2-Chat, has been engineered explicitly for dialogue-centric applications. Benchmarking revealed Llama 2's superior performance over most extant open-source chat models. 

Metadata: {'title': "Beyond GPT-4: What's New?", 'url': 'https://pub.towardsai.net/beyond-gpt-4-whats-new-cbd61a448eb9#dda8', 'source_name': 'towards_ai'}


# Transforming


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter

# Define the splitter object that split the text into segments with 512 tokens,
# with a 128 overlap between the segments.
text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

# Create the pipeline to apply the transformation (splitting and embedding) on each chunk,
# and store the transformed text in the chroma vector store.
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(model = 'text-embedding-3-small'),
    ],
    vector_store=vector_store,
)

# Run the transformation pipeline.
b = pipeline.run(documents=documents, show_progress=True)

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

# Load Indexes


In [ ]:
# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = db.get_or_create_collection("mini-llama-articles")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store)

In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    max_output_tokens=512,
    temperature=1,
)

llm = GoogleGenAI(
    model="gemini-2.5-flash",
    generation_config=config,
    )

query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

In [ ]:
res = query_engine.query("How many parameters does LLaMA 2 model have?")

In [ ]:
print("Response (compact mode):")
print(res.response)

Response (compact mode):
The Llama 2 model is available in various sizes, including 7 billion, 13 billion, 34 billion, and 70 billion parameters.


In [ ]:
# Show the source nodes used to generate the response
print(f"\nSource nodes retrieved ({len(res.source_nodes)}):")
for i, src in enumerate(res.source_nodes):
    print(f"\n  Node {i+1}")
    print(f"  Node ID : {src.node_id}")
    print(f"  Title   : {src.metadata['title']}")
    print(f"  Score   : {src.score:.4f}")
    print(f"  Text    : {src.text[:200]}...")
    print("  " + "-" * 40)


Source nodes retrieved (5):

  Node 1
  Node ID : 2a424c1a-f5ad-421d-b517-bdac6cc653a3
  Title   : Fine-Tuning a Llama-2 7B Model for Python Code Generation
  Score   : 0.3618
  Text    : only fine-tuning a small number of additional parameters, with virtually all model parameters remaining frozen. PEFT has been found to produce good generalization with relatively low-volume datasets. ...
  ----------------------------------------

  Node 2
  Node ID : cbcb0c00-3d58-45c0-b429-3f041b30694f
  Title   : Exploring Large Language Models -Part 3
  Score   : 0.3595
  Text    : The results confirm that the model learns via Instruct tuning, not only the fed questions but other details and relations of the domain. Problems with hallucinations remain (Bordor, Lila characters wh...
  ----------------------------------------

  Node 3
  Node ID : cdd47b6d-b61e-4a95-9fa9-35a692b31eaa
  Title   : Fine-Tuning a Llama-2 7B Model for Python Code Generation
  Score   : 0.3588
  Text    : weights As we m

# Response Modes


LlamaIndex's query engine supports several response synthesis modes. Each makes different
trade-offs between cost (number of LLM calls) and answer quality:

| Mode | LLM calls | How it works |
|---|---|---|
| `compact` *(default)* | 1 | Fits all retrieved chunks into one prompt; single LLM call |
| `refine` | N (one per chunk) | Starts with the first chunk, iteratively refines the answer using each subsequent chunk |
| `tree_summarize` | ~log(N) | Recursively summarizes chunk pairs into a single final answer |
| `no_text` | 0 | Retrieves nodes only — no LLM call; useful for debugging retrieval |

See the [LlamaIndex response modes docs](https://docs.llamaindex.ai/en/stable/module_guides/querying/response_synthesizers/root.html) for the full list.

> **Note:** With a small dataset, all modes may return identical answers.
> The differences are most visible with larger corpora and longer documents.


In [ ]:
query_engine = index.as_query_engine(response_mode="refine", llm=llm)
#query_engine = index.as_query_engine(response_mode="tree_summarize")

In [ ]:
res = query_engine.query("How many parameters does LLaMA 2 model have?")

In [ ]:
print("Response (refine mode):")
print(res.response)

Response (refine mode):
The LLaMA 2 model has 7 billion parameters.


In [ ]:
print(f"\nSource nodes retrieved ({len(res.source_nodes)}):")
for i, src in enumerate(res.source_nodes):
    print(f"\n  Node {i+1}")
    print(f"  Title   : {src.metadata['title']}")
    print(f"  Score   : {src.score:.4f}")
    print(f"  Text    : {src.text[:200]}...")
    print("  " + "-" * 40)


Source nodes retrieved (2):

  Node 1
  Title   : Fine-Tuning a Llama-2 7B Model for Python Code Generation
  Score   : 0.3618
  Text    : only fine-tuning a small number of additional parameters, with virtually all model parameters remaining frozen. PEFT has been found to produce good generalization with relatively low-volume datasets. ...
  ----------------------------------------

  Node 2
  Title   : Exploring Large Language Models -Part 3
  Score   : 0.3595
  Text    : The results confirm that the model learns via Instruct tuning, not only the fed questions but other details and relations of the domain. Problems with hallucinations remain (Bordor, Lila characters wh...
  ----------------------------------------


The `no_text` mode will retrieve the documents, but will not send the request to the API to synthesize the final response. It is a great approach to debug the retrieved documents.


In [ ]:
query_engine = index.as_query_engine(response_mode="no_text", llm=llm)
res = query_engine.query("How many parameters does LLaMA 2 model have?")

In [ ]:
# no_text mode does not generate an LLM response — useful for debugging retrieval
print(f"Response (no_text mode): {repr(res.response)}")
print("(no_text mode skips the LLM call — only retrieval runs)")

Response (no_text mode): ''
(no_text mode skips the LLM call — only retrieval runs)


In [ ]:
# Inspect the retrieved nodes to verify retrieval quality
print(f"\nSource nodes retrieved ({len(res.source_nodes)}):")
for i, src in enumerate(res.source_nodes):
    print(f"\n  Node {i+1}")
    print(f"  Title   : {src.metadata['title']}")
    print(f"  Score   : {src.score:.4f}")
    print(f"  Text    : {src.text[:200]}...")
    print("  " + "-" * 40)


Source nodes retrieved (2):

  Node 1
  Title   : Fine-Tuning a Llama-2 7B Model for Python Code Generation
  Score   : 0.3618
  Text    : only fine-tuning a small number of additional parameters, with virtually all model parameters remaining frozen. PEFT has been found to produce good generalization with relatively low-volume datasets. ...
  ----------------------------------------

  Node 2
  Title   : Exploring Large Language Models -Part 3
  Score   : 0.3595
  Text    : The results confirm that the model learns via Instruct tuning, not only the fed questions but other details and relations of the domain. Problems with hallucinations remain (Bordor, Lila characters wh...
  ----------------------------------------


---
## Optional: Custom Prompt Template

By default, LlamaIndex uses a built-in prompt template that combines retrieved
context with the user's question. You can replace it with a custom `PromptTemplate`
to control exactly how the LLM receives and uses the retrieved information.

In [ ]:
from llama_index.core import PromptTemplate

# Define a custom prompt that instructs the LLM more precisely
custom_qa_template = PromptTemplate(
    "You are an expert AI researcher. Use ONLY the context below to answer the question.\n"
    "If the answer is not in the context, say 'I don't have enough information to answer this.'\n\n"
    "Context:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n\n"
    "Question: {query_str}\n\n"
    "Answer (be concise and cite specific details from the context):"
)

# Apply the custom template to a new query engine instance
custom_query_engine = index.as_query_engine(
    llm=llm,
    text_qa_template=custom_qa_template,
)

custom_res = custom_query_engine.query("What are the safety features of LLaMA 2?")
print("Custom template response:")
print(custom_res.response)
print(f"\nSource nodes used: {len(custom_res.source_nodes)}")

Custom template response:
LLaMA 2 demonstrates exceptionally low AI safety violation percentages, even surpassing ChatGPT in safety benchmarks. Meta prioritized safety and alignment in its design. To achieve a balance between helpfulness and safety, Meta utilized two reward models: one for helpfulness and another for safety.

Source nodes used: 2


---
## Optional: All Response Modes — Side-by-Side Comparison

Running the same question through `compact`, `refine`, and `tree_summarize` modes
lets you compare how each synthesis strategy affects the answer.

In [ ]:
COMPARISON_QUERY = "What makes LLaMA 2 different from the original LLaMA model?"
modes = ["compact", "refine", "tree_summarize"]

print(f"Query: {COMPARISON_QUERY}\n")
print("=" * 70)

for mode in modes:
    try:
        eng = index.as_query_engine(response_mode=mode, llm=llm)
        r = eng.query(COMPARISON_QUERY)
        print(f"\n[{mode.upper()}]")
        response_text = r.response if r.response else "(no response generated)"
        print(response_text[:600])
        print(f"(Used {len(r.source_nodes)} source node(s))")
    except Exception as e:
        print(f"\n[{mode.upper()}] Error: {e}")

Query: What makes LLaMA 2 different from the original LLaMA model?


[COMPACT]
LLaMA 2 has several features that differentiate it from its predecessor. It incorporates Ghost Attention, which allows the model to maintain conversational continuity by remembering initial instructions over multiple interactions, leading to more coherent and consistent responses. Additionally, LLaMA 2 boasts a temporal capability, enabling it to organize information based on time relevance and consider event dates when responding to questions, thereby delivering more contextually accurate answers. Unlike the original LLaMA, which was intended solely for researchers and academics, LLaMA 2 is o
(Used 2 source node(s))

[REFINE]
LLaMA 2 stands out due to several key advancements. It features Ghost Attention, which significantly enhances conversational continuity by allowing the model to remember initial instructions across multiple interactions, leading to more coherent responses. Another groundbreaking capabi